# Optativa III: Ciencia de Datos
## Semana 4 — Transformación de datos

**Universidad de Especialidades UNE · Plantel Centro**
**Ingeniería en Computación · 9.º semestre · Semana 4 · Sesión 2**

---

### Propósito

Una vez que los datos están limpios, hay que **transformarlos** para que los algoritmos puedan aprovecharlos. En esta actividad dominarás las cinco técnicas centrales de transformación de datos:

- **Normalización** (Min-Max) — llevar variables a un rango común [0, 1].
- **Estandarización** (Z-score) — centrar en media 0 y desviación 1.
- **Codificación de variables categóricas** — convertir texto en números (ordinal, one-hot, label encoding).
- **Feature engineering** — crear variables nuevas más informativas.
- **Manejo de fechas** — extraer información útil de columnas temporales.

Al final construirás un **pipeline de transformación reproducible** y prepararás un **dataset limpio con su diccionario de datos**.

La actividad tiene dos partes:

- **PARTE A — Ejemplo guiado por el profesor.** Transformaremos juntos un dataset de empleados, paso a paso.
- **PARTE B — Tu turno.** Aplicarás las técnicas de forma autónoma sobre otros datos y construirás tu propio pipeline.

Responderás **50 preguntas ✍️** modificando código, analizando resultados e interpretando.

### Los datasets (súbelos a Colab antes de empezar)

- `empleados.csv` — datos de 500 empleados (numéricas, categóricas y fechas). Para la Parte A.
- `amazon.csv` — productos de Amazon (ya limpio). Para la Parte B.

Sube ambos con el panel de archivos de Colab (icono de carpeta → subir).

### Cómo trabajar

Ejecuta cada celda con **Shift + Enter** en orden, responde las preguntas ✍️, completa los retos y, al final, guarda una copia en Drive y entrégala en Classroom.


---
# PARTE A — Ejemplo guiado por el profesor

## Paso 0 — Carga y exploración


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler, LabelEncoder

pd.set_option("display.max_columns", None)

df = pd.read_csv("empleados.csv")
print("Dimensiones:", df.shape)
df.head()

In [ ]:
# Identificamos tipos de variable
print("Tipos de datos:")
print(df.dtypes)
print("\nVariables numericas:", df.select_dtypes(include="number").columns.tolist())
print("Variables categoricas:", df.select_dtypes(include="object").columns.tolist())

> ✍️ **1.** ¿Cuántos empleados y cuántas variables tiene el dataset? Enumera qué variables son numéricas y cuáles categóricas.

**Respuesta:** Hay **500 empleados y 11 variables**. Numéricas: `empleado_id`, `edad`, `salario_mensual`, `horas_semana`, `proyectos_completados`, `satisfaccion`. Categóricas (texto): `departamento`, `nivel`, `ciudad`, `educacion`, más `fecha_ingreso`, que es una fecha guardada como texto y habrá que convertirla.

> ✍️ **2.** ¿Por qué necesitamos "transformar" los datos si ya están limpios? ¿Qué problema tiene alimentar a un algoritmo con `salario_mensual` (decenas de miles) y `satisfaccion` (1 a 5) en sus escalas originales?

**Respuesta:** Limpiar corrige errores; transformar pone los datos en la **forma que el algoritmo necesita**. Si mezclamos `salario_mensual` (21,533–128,668) con `satisfaccion` (1–5) sin escalar, cualquier algoritmo basado en distancias o magnitudes queda **dominado por el salario**: una diferencia de \$1,000 pesaría cientos de veces más que un punto completo de satisfacción, no porque importe más, sino por pura escala. Además los algoritmos no entienden texto: `nivel` o `departamento` deben codificarse como números para poder usarse.

> ✍️ **3.** La columna `fecha_ingreso` aparece como `object` (texto). ¿Por qué pandas no la reconoce automáticamente como fecha? ¿Qué tendremos que hacer con ella?

**Respuesta:** Porque `read_csv` solo infiere números; no intenta adivinar fechas (hay demasiados formatos posibles: `2016-08-18`, `18/08/2016`, etc.), así que la deja como texto. Habrá que convertirla con `pd.to_datetime()`; solo entonces podremos extraer año, mes o día de la semana y calcular la antigüedad restando fechas.


## Paso 1 — Normalización (Min-Max Scaling)

La **normalización** transforma una variable a un rango fijo, típicamente [0, 1], con la fórmula:

$$x_{norm} = \frac{x - x_{min}}{x_{max} - x_{min}}$$

Es útil cuando quieres que todas las variables tengan el mismo peso en algoritmos basados en distancias.


In [ ]:
# Normalizamos el salario a [0, 1]
scaler_mm = MinMaxScaler()
df["salario_norm"] = scaler_mm.fit_transform(df[["salario_mensual"]])

print("Salario original -> min: {:.0f}  max: {:.0f}".format(
    df["salario_mensual"].min(), df["salario_mensual"].max()))
print("Salario normalizado -> min: {:.3f}  max: {:.3f}".format(
    df["salario_norm"].min(), df["salario_norm"].max()))

# Visualizamos antes y despues
fig, axes = plt.subplots(1, 2, figsize=(13,4))
axes[0].hist(df["salario_mensual"], bins=30, color="#1e2a6b", alpha=0.7)
axes[0].set_title("Salario ORIGINAL"); axes[0].set_xlabel("Pesos")
axes[1].hist(df["salario_norm"], bins=30, color="#e2231a", alpha=0.7)
axes[1].set_title("Salario NORMALIZADO [0,1]"); axes[1].set_xlabel("Valor")
plt.tight_layout(); plt.show()

> ✍️ **4.** Tras normalizar, ¿cuál es el valor mínimo y máximo de `salario_norm`? ¿A qué salario original corresponde el valor 0 y a cuál el valor 1?

**Respuesta:** El mínimo es exactamente **0** y el máximo exactamente **1** (así funciona Min-Max por construcción). El 0 corresponde al salario más bajo del dataset, **\$21,533**, y el 1 al más alto, **\$128,668**; todos los demás quedan en proporción entre ambos extremos.

> ✍️ **5.** Compara los dos histogramas. ¿Cambió la **forma** de la distribución al normalizar, o solo la **escala** del eje X? Explica.

**Respuesta:** Solo cambió la **escala** del eje X: la forma es idéntica. La normalización es una transformación lineal (restar el mínimo y dividir entre el rango), así que cada dato conserva su posición relativa; el histograma es el mismo dibujo con otras etiquetas en el eje. Ni el sesgo ni los picos de la distribución se alteran.

> ✍️ **6.** La normalización Min-Max es sensible a los valores atípicos. Si hubiera un salario de 10 millones (outlier), ¿qué pasaría con el resto de los valores normalizados? (Pista: piensa dónde quedaría el máximo.)

**Respuesta:** El outlier se convertiría en el nuevo máximo (valor 1) y el denominador de la fórmula se dispararía a casi 10 millones. Todos los salarios normales (21 mil a 128 mil) quedarían **aplastados en un rango diminuto cerca de 0** (aproximadamente entre 0.001 y 0.011), perdiendo casi toda su capacidad de distinguirse entre sí. Un solo valor extremo arruina la escala completa; por eso, con outliers, conviene tratarlos antes o usar estandarización.


### 🔧 Reto 1 — Normaliza otra variable

Normaliza la columna `edad` a [0, 1] creando `edad_norm`. Verifica su mínimo y máximo.


In [ ]:
# RETO 1 resuelto: normalizar edad
df["edad_norm"] = scaler_mm.fit_transform(df[["edad"]])

print(f"Edad original -> min: {df['edad'].min()}  max: {df['edad'].max()}")
print(df["edad_norm"].describe().round(3))

> ✍️ **7.** ¿Cuál es la edad mínima y máxima del dataset? ¿A qué valor normalizado corresponde un empleado de edad promedio?

**Respuesta:** La edad mínima es **22** años y la máxima **59**. La edad promedio es 40.0 años, que normalizada corresponde a (40.04 − 22) / (59 − 22) ≈ **0.49**: prácticamente el punto medio del rango, lo cual indica que el promedio está centrado entre los extremos de edad.


## Paso 2 — Estandarización (Z-score)

La **estandarización** centra la variable en media 0 y desviación estándar 1:

$$z = \frac{x - \mu}{\sigma}$$

Un valor de z = 2 significa "2 desviaciones estándar por encima de la media". Es la transformación preferida para muchos modelos estadísticos y de machine learning.


In [ ]:
# Estandarizamos el salario (Z-score)
scaler_z = StandardScaler()
df["salario_z"] = scaler_z.fit_transform(df[["salario_mensual"]])

print("Salario estandarizado:")
print("  media: {:.4f}  (debe ser ~0)".format(df["salario_z"].mean()))
print("  desviacion: {:.4f}  (debe ser ~1)".format(df["salario_z"].std()))
print("  min: {:.2f}  max: {:.2f}".format(df["salario_z"].min(), df["salario_z"].max()))

# Un empleado con salario muy alto tendra z grande
ejemplo = df.nlargest(1, "salario_mensual")[["salario_mensual", "salario_z"]]
print("\nEmpleado mejor pagado:")
print(ejemplo.to_string(index=False))

> ✍️ **8.** Tras estandarizar, ¿cuál es la media y la desviación estándar de `salario_z`? ¿Por qué siempre dan 0 y 1?

> ✍️ **9.** El empleado mejor pagado tiene un `salario_z` de cierto valor. Interprétalo: ¿cuántas desviaciones estándar por encima del promedio está?

> ✍️ **10.** ¿Cuál es la diferencia clave entre **normalización** (Min-Max) y **estandarización** (Z-score)? ¿En qué caso usarías cada una?

> ✍️ **11.** A diferencia de la normalización, la estandarización NO acota los valores a un rango fijo. ¿Entre qué valores puede moverse un z-score en la práctica? ¿Puede ser negativo?


### 🔧 Reto 2 — Estandariza y compara

Estandariza la columna `horas_semana` creando `horas_z`. Luego encuentra al empleado con el z-score más alto (el que más horas trabaja) usando `df.nlargest()`.


In [ ]:
# RETO: estandariza horas_semana y encuentra el maximo
# df["horas_z"] = scaler_z.fit_transform(df[["horas_semana"]])
# print(df.nlargest(3, "horas_z")[["horas_semana", "horas_z"]])


> ✍️ **12.** ¿Cuántas horas trabaja el empleado con el z-score más alto y cuál es ese z-score? ¿Está muy lejos de la media?


## Paso 3 — Codificación de variables categóricas

Los algoritmos no entienden texto. Hay que convertir las categorías en números. Existen tres técnicas según el tipo de variable.

### 3.1 — Codificación ORDINAL (para categorías con orden)

`nivel` (Junior < Semi-Senior < Senior < Lead) y `educacion` tienen un **orden natural**. Les asignamos números que respeten ese orden.


In [ ]:
# Codificacion ordinal: respetamos la jerarquia
orden_nivel = {"Junior": 1, "Semi-Senior": 2, "Senior": 3, "Lead": 4}
df["nivel_cod"] = df["nivel"].map(orden_nivel)

orden_educacion = {"Técnico": 1, "Licenciatura": 2, "Maestría": 3, "Doctorado": 4}
df["educacion_cod"] = df["educacion"].map(orden_educacion)

print("Codificacion de nivel:")
print(df[["nivel", "nivel_cod"]].drop_duplicates().sort_values("nivel_cod").to_string(index=False))

> ✍️ **13.** ¿Por qué asignamos números en orden (1, 2, 3, 4) a `nivel` en lugar de números al azar? ¿Qué información se perdería si usáramos códigos aleatorios?

> ✍️ **14.** ¿Qué pasaría si a `educacion` (que SÍ tiene orden) la codificáramos con one-hot (sin orden)? ¿Perderíamos información?


### 3.2 — Codificación ONE-HOT (para categorías sin orden)

`departamento` y `ciudad` **no tienen orden** (Ventas no es "mayor" que Marketing). Para ellas usamos **one-hot encoding**: creamos una columna binaria (0/1) por cada categoría.


In [ ]:
# One-hot encoding del departamento
dummies_dep = pd.get_dummies(df["departamento"], prefix="dep")
print("Columnas creadas por one-hot:")
print(dummies_dep.columns.tolist())
print("\nEjemplo (primeras filas):")
print(dummies_dep.head())

# Unimos al dataframe
df = pd.concat([df, dummies_dep], axis=1)
print("\nDimensiones tras one-hot:", df.shape)

> ✍️ **15.** ¿Cuántas columnas nuevas creó el one-hot encoding de `departamento`? ¿Por qué esa cantidad?

> ✍️ **16.** En una fila donde `departamento = "Ventas"`, ¿qué valor tendrá la columna `dep_Ventas` y qué valor las demás columnas `dep_...`?

> ✍️ **17.** ¿Por qué NO usamos one-hot para `nivel`? ¿Qué problema traería convertir una variable ordinal en columnas one-hot independientes?

> ✍️ **18.** El one-hot puede crear muchísimas columnas si la variable tiene muchas categorías (ej. 1000 ciudades). ¿Qué problema práctico genera esto? (Investiga el término "maldición de la dimensionalidad".)


### 3.3 — LABEL ENCODING (asignar un número a cada categoría)

`LabelEncoder` asigna un entero a cada categoría. Es rápido, pero **introduce un orden artificial** que no siempre es deseable.


In [ ]:
# Label encoding de ciudad
le = LabelEncoder()
df["ciudad_cod"] = le.fit_transform(df["ciudad"])

print("Mapeo de label encoding:")
for i, clase in enumerate(le.classes_):
    print(f"  {clase} -> {i}")

> ✍️ **19.** ¿Qué número le asignó el label encoder a cada ciudad? ¿En qué orden las numeró (pista: alfabético)?

> ✍️ **20.** El label encoding le da a `ciudad` los valores 0,1,2,3,4. Un algoritmo podría interpretar erróneamente que "Querétaro (4) es 4 veces Guadalajara (1)". ¿Por qué es esto un problema para una variable SIN orden? ¿Cuándo sería preferible one-hot?


### 🔧 Reto 3 — Codifica `educacion` con one-hot

Aunque `educacion` tiene orden, practica el one-hot encoding sobre ella para comparar. Crea las columnas dummy y cuenta cuántas son.


In [ ]:
# RETO: one-hot de educacion
# dummies_edu = pd.get_dummies(df["educacion"], prefix="edu")
# print(dummies_edu.columns.tolist())


> ✍️ **21.** ¿Cuántas columnas creó el one-hot de `educacion`? Comparado con la codificación ordinal (una sola columna), ¿cuál ocupa menos espacio? ¿Cuál conserva el orden?


## Paso 4 — Manejo de fechas (feature engineering temporal)

Una columna de fecha es una mina de información. La convertimos a tipo fecha y extraemos variables útiles.


In [ ]:
# Convertir a tipo datetime
df["fecha_ingreso"] = pd.to_datetime(df["fecha_ingreso"])
print("Tipo tras conversion:", df["fecha_ingreso"].dtype)

# Extraer componentes de la fecha
df["anio_ingreso"] = df["fecha_ingreso"].dt.year
df["mes_ingreso"] = df["fecha_ingreso"].dt.month
df["dia_semana_ingreso"] = df["fecha_ingreso"].dt.day_name()

# Calcular antiguedad en años (feature derivada muy util)
fecha_referencia = pd.Timestamp("2025-01-01")
df["antiguedad_anios"] = ((fecha_referencia - df["fecha_ingreso"]).dt.days / 365.25).round(1)

print(df[["fecha_ingreso", "anio_ingreso", "mes_ingreso", "antiguedad_anios"]].head())

> ✍️ **22.** ¿Qué hace `pd.to_datetime()`? ¿Por qué es necesario antes de poder extraer el año o el mes?

> ✍️ **23.** ¿Qué componentes extrajimos de la fecha? Menciona al menos otros dos que podrías extraer (ej. trimestre, si es fin de semana).

> ✍️ **24.** La `antiguedad_anios` es una **feature derivada**: no estaba en los datos, la calculamos. ¿Por qué es más útil para un análisis de RH que la fecha de ingreso cruda?

> ✍️ **25.** ¿Cuál es la antigüedad promedio de los empleados? Calcúlala en una celda. ¿Y el empleado más antiguo cuántos años lleva?


In [ ]:
# Celda para la P25
# print("Antiguedad promedio:", df["antiguedad_anios"].mean().round(1))
# print("Empleado mas antiguo:", df["antiguedad_anios"].max())


## Paso 5 — Feature engineering (crear variables nuevas)

El **feature engineering** es el arte de crear variables que capturen mejor la información. Un buen feature puede mejorar un modelo más que un algoritmo sofisticado.


In [ ]:
# Feature 1: salario por proyecto completado (productividad economica)
df["salario_por_proyecto"] = (df["salario_mensual"] /
                              df["proyectos_completados"].replace(0, 1)).round(2)

# Feature 2: categoria de antiguedad (novato / establecido / veterano)
df["categoria_antiguedad"] = pd.cut(df["antiguedad_anios"],
                                    bins=[0, 2, 5, 100],
                                    labels=["Novato", "Establecido", "Veterano"])

# Feature 3: indicador binario de alto desempeño
df["alto_desempeno"] = ((df["proyectos_completados"] > df["proyectos_completados"].median()) &
                        (df["satisfaccion"] > 3.5)).astype(int)

print("Nuevas features creadas:")
print(df[["salario_por_proyecto", "categoria_antiguedad", "alto_desempeno"]].head())
print("\nDistribucion de categoria_antiguedad:")
print(df["categoria_antiguedad"].value_counts())

> ✍️ **26.** Explica qué representa cada una de las tres features nuevas. ¿Cuál te parece más útil para predecir el desempeño de un empleado?

> ✍️ **27.** La feature `alto_desempeno` combina DOS condiciones. ¿Cuáles son? ¿Por qué combinar variables puede crear un indicador más potente que cada una por separado?

> ✍️ **28.** ¿Cuántos empleados quedaron en cada `categoria_antiguedad`? ¿Qué grupo es el más numeroso?

> ✍️ **29.** El feature engineering requiere **conocimiento del dominio**. Propón UNA feature nueva que se te ocurra para este dataset de RH y explica por qué sería útil.


### 🔧 Reto 4 — Crea tu propia feature

Crea una feature `salario_anual` (salario mensual × 12) y otra `es_senior` (1 si el nivel es Senior o Lead, 0 si no). Usa `.isin()` para la segunda.


In [ ]:
# RETO: crea salario_anual y es_senior
# df["salario_anual"] = df["salario_mensual"] * 12
# df["es_senior"] = df["nivel"].isin(["Senior", "Lead"]).astype(int)
# print(df[["salario_anual", "es_senior"]].head())


> ✍️ **30.** ¿Cuántos empleados son "senior" (Senior o Lead) según tu feature? ¿Qué porcentaje del total representan?


## Paso 6 — Pipeline de transformación reproducible

Un **pipeline** encapsula todos los pasos de transformación en una secuencia ordenada y reutilizable. Esto garantiza **reproducibilidad**: aplicar exactamente las mismas transformaciones a datos nuevos.


In [ ]:
def pipeline_transformacion(datos):
    """Aplica todas las transformaciones en orden. Recibe un DataFrame crudo
    y devuelve uno transformado, listo para analisis o modelado."""
    d = datos.copy()

    # 1. Fechas
    d["fecha_ingreso"] = pd.to_datetime(d["fecha_ingreso"])
    fecha_ref = pd.Timestamp("2025-01-01")
    d["antiguedad_anios"] = ((fecha_ref - d["fecha_ingreso"]).dt.days / 365.25).round(1)

    # 2. Codificacion ordinal
    d["nivel_cod"] = d["nivel"].map({"Junior":1, "Semi-Senior":2, "Senior":3, "Lead":4})
    d["educacion_cod"] = d["educacion"].map({"Técnico":1, "Licenciatura":2, "Maestría":3, "Doctorado":4})

    # 3. Estandarizacion de numericas
    for col in ["salario_mensual", "edad", "horas_semana"]:
        d[col + "_z"] = StandardScaler().fit_transform(d[[col]])

    # 4. One-hot de nominales
    d = pd.concat([d, pd.get_dummies(d["departamento"], prefix="dep")], axis=1)

    # 5. Feature derivada
    d["salario_anual"] = d["salario_mensual"] * 12

    return d

# Aplicamos el pipeline al dataset original crudo
df_crudo = pd.read_csv("empleados.csv")
df_transformado = pipeline_transformacion(df_crudo)
print("Dataset crudo:", df_crudo.shape)
print("Dataset transformado:", df_transformado.shape)
print("\nNuevas columnas:", [c for c in df_transformado.columns if c not in df_crudo.columns])

> ✍️ **31.** ¿Cuántas columnas nuevas agregó el pipeline? ¿Por qué encapsular las transformaciones en una función es mejor que ejecutarlas sueltas una por una?

> ✍️ **32.** ¿Por qué usamos `datos.copy()` al inicio de la función? ¿Qué pasaría si modificáramos el DataFrame original directamente?

> ✍️ **33.** La **reproducibilidad** es clave en ciencia de datos. Si mañana llegan 100 empleados nuevos, ¿cómo aplicarías exactamente las mismas transformaciones? ¿Por qué el pipeline lo facilita?


### 🔧 Reto 5 — Amplía el pipeline

Agrega al pipeline un paso que cree la feature `categoria_antiguedad` (Novato/Establecido/Veterano) con `pd.cut()`. Vuelve a ejecutarlo y verifica que aparezca.


In [ ]:
# RETO: modifica pipeline_transformacion para agregar categoria_antiguedad
# (copia la funcion, agrega el paso con pd.cut y vuelve a aplicarla)


> ✍️ **34.** Tras ampliar el pipeline, ¿aparece la nueva columna? ¿Por qué es importante que el pipeline sea fácil de modificar y extender?


## Paso 7 — Verificación del dataset transformado


In [ ]:
# Comparamos escalas antes y despues de transformar
print("=== COMPARACION DE ESCALAS ===")
print("\nVariables ORIGINALES (escalas muy distintas):")
print(df[["salario_mensual", "edad", "satisfaccion"]].describe().loc[["mean", "std", "min", "max"]])

print("\nVariables ESTANDARIZADAS (misma escala):")
cols_z = ["salario_z"]
if "salario_mensual_z" in df_transformado.columns:
    print(df_transformado[["salario_mensual_z", "edad_z", "horas_semana_z"]].describe().loc[["mean", "std"]])

> ✍️ **35.** Compara las escalas: antes de transformar, ¿qué variable tenía los valores más grandes y cuál los más pequeños? Después de estandarizar, ¿todas quedaron en la misma escala?

> ✍️ **36.** Grafica un histograma de `salario_mensual_z` del dataset transformado. ¿Está centrado en cero?


In [ ]:
# Celda para la P36
# plt.hist(df_transformado["salario_mensual_z"], bins=30, color="#1e2a6b", alpha=0.7)
# plt.axvline(0, color="#e2231a", linestyle="--")
# plt.title("Salario estandarizado"); plt.show()


---
# PARTE B — Tu turno (dataset de Amazon)

Ahora aplicarás las técnicas de transformación de forma autónoma sobre el dataset de productos de **Amazon**.

## Paso 8 — Carga y diagnóstico


In [ ]:
amazon = pd.read_csv("amazon.csv")
print("Dimensiones:", amazon.shape)
print("\nColumnas:", amazon.columns.tolist())
amazon.head()

> ✍️ **37.** ¿Qué variables numéricas tiene el dataset de Amazon que podrías normalizar o estandarizar? ¿Qué variable es categórica y se podría codificar?

> ✍️ **38.** ¿Por qué convendría estandarizar `actual_price` y `rating_count` antes de, por ejemplo, calcular distancias entre productos? (Pista: sus escalas son muy distintas.)


### 🔧 Reto 6 — Normaliza y estandariza precios

Crea `precio_norm` (normalización Min-Max de `actual_price`) y `precio_z` (estandarización). Compara sus rangos.


In [ ]:
# RETO: normaliza y estandariza actual_price
# amazon["precio_norm"] = MinMaxScaler().fit_transform(amazon[["actual_price"]])
# amazon["precio_z"] = StandardScaler().fit_transform(amazon[["actual_price"]])
# print(amazon[["precio_norm", "precio_z"]].describe())


> ✍️ **39.** ¿Cuál es el rango de `precio_norm` y el de `precio_z`? ¿Cuál tiene media 0?

> ✍️ **40.** El producto más caro, ¿qué valor tiene en `precio_norm`? ¿Y aproximadamente en `precio_z`? Interpreta el z-score.


### 🔧 Reto 7 — Codifica la categoría

La columna `category` es categórica nominal. Aplícale one-hot encoding y cuenta cuántas columnas nuevas se crean.


In [ ]:
# RETO: one-hot encoding de category
# dummies_cat = pd.get_dummies(amazon["category"], prefix="cat")
# print("Columnas creadas:", len(dummies_cat.columns))
# print(dummies_cat.columns.tolist())


> ✍️ **41.** ¿Cuántas categorías distintas hay y cuántas columnas one-hot se crearon? ¿Es one-hot la mejor opción aquí, o habría demasiadas columnas?

> ✍️ **42.** Si `category` tuviera 200 categorías distintas, ¿seguirías usando one-hot? ¿Qué alternativa considerarías?


### 🔧 Reto 8 — Feature engineering sobre Amazon

Crea la feature `es_popular` (1 si `rating_count` supera la mediana, 0 si no) y `precio_x_rating` (`actual_price` × `rating`). Interpreta qué mide cada una.


In [ ]:
# RETO: crea es_popular y precio_x_rating
# amazon["es_popular"] = (amazon["rating_count"] > amazon["rating_count"].median()).astype(int)
# amazon["precio_x_rating"] = (amazon["actual_price"] * amazon["rating"]).round(2)
# print(amazon[["es_popular", "precio_x_rating"]].head())


> ✍️ **43.** ¿Qué mide la feature `es_popular`? ¿Cuántos productos son "populares" según tu definición?

> ✍️ **44.** ¿Tiene sentido de negocio la feature `precio_x_rating`? ¿Qué tipo de producto tendría un valor alto en ella? Reflexiona si esta feature es realmente útil o es artificial.


### 🔧 Reto 9 — Pipeline para Amazon

Escribe una función `pipeline_amazon(datos)` que aplique: (1) estandarización de `actual_price` y `rating_count`, (2) one-hot de `category`, y (3) la feature `es_popular`. Aplícala y verifica las nuevas columnas.


In [ ]:
# RETO: construye tu pipeline para Amazon
# def pipeline_amazon(datos):
#     d = datos.copy()
#     d["precio_z"] = StandardScaler().fit_transform(d[["actual_price"]])
#     d["count_z"] = StandardScaler().fit_transform(d[["rating_count"]])
#     d = pd.concat([d, pd.get_dummies(d["category"], prefix="cat")], axis=1)
#     d["es_popular"] = (d["rating_count"] > d["rating_count"].median()).astype(int)
#     return d
#
# amazon_transformado = pipeline_amazon(amazon)
# print(amazon_transformado.shape)


> ✍️ **45.** ¿Cuántas columnas tiene el dataset de Amazon tras aplicar tu pipeline? ¿Por qué un pipeline es especialmente valioso cuando trabajas con un dataset grande y repetitivo?


## Paso 9 — Análisis del dataset transformado


In [ ]:
# Analisis: salario promedio por nivel (usando el dataset de empleados transformado)
print("Salario promedio por nivel:")
print(df.groupby("nivel")["salario_mensual"].mean().round(2).sort_values())

# Grafico
fig, axes = plt.subplots(1, 2, figsize=(13,5))
df.groupby("nivel")["salario_mensual"].mean().sort_values().plot(
    kind="bar", ax=axes[0], color="#1e2a6b")
axes[0].set_title("Salario promedio por nivel"); axes[0].set_ylabel("Pesos")
df["categoria_antiguedad"].value_counts().plot(kind="pie", ax=axes[1], autopct="%1.0f%%")
axes[1].set_title("Distribucion por antiguedad"); axes[1].set_ylabel("")
plt.tight_layout(); plt.show()

> ✍️ **46.** ¿Qué nivel tiene el salario promedio más alto? ¿La codificación ordinal de `nivel` refleja correctamente esta jerarquía salarial?

> ✍️ **47.** ¿Cómo te ayudaron las transformaciones (codificación, features) a producir este análisis? ¿Podrías haber agrupado por nivel sin transformarlo?

> ✍️ **48.** De todas las técnicas de transformación (normalización, estandarización, codificación, feature engineering, fechas), ¿cuál te pareció más útil y por qué?

> ✍️ **49.** Un compañero normaliza los datos DESPUÉS de dividir en entrenamiento y prueba, usando el máximo de TODO el dataset. ¿Por qué esto es un error (fuga de datos / data leakage)? ¿Cómo debería hacerse?

> ✍️ **50 (reflexión).** En tu carrera de Ingeniería en Computación, ¿en qué proyecto real necesitarías transformar datos antes de analizarlos o modelarlos? Describe qué transformaciones aplicarías (ej. datos de sensores, logs con timestamps, categorías de usuarios).


---
# TAREA — Dataset limpio y documentado (con diccionario de datos)

Como tarea final, prepara un **dataset transformado, limpio y documentado**. Entrégalo junto con este cuaderno.

### Instrucciones

1. Toma el dataset de **empleados** (o el de Amazon) ya transformado con tu pipeline.
2. Guarda el resultado en un archivo CSV limpio con `df_transformado.to_csv("dataset_limpio.csv", index=False)`.
3. Crea un **diccionario de datos**: una tabla que documente CADA columna del dataset final.

### El diccionario de datos debe incluir, por cada columna:

| Columna | Tipo | Descripción | Transformación aplicada | Valores posibles / rango |
|---|---|---|---|---|
| salario_mensual | float | Salario mensual en pesos | Ninguna (original) | 10,000 – 90,000 |
| salario_z | float | Salario estandarizado | Z-score | media 0, desv 1 |
| nivel_cod | int | Nivel jerárquico codificado | Ordinal (1-4) | 1=Junior ... 4=Lead |
| ... | ... | ... | ... | ... |

Documenta **todas** las columnas del dataset transformado. Puedes crear el diccionario en una celda de código (como DataFrame) o en una celda de texto (tabla Markdown).


In [ ]:
# TAREA: genera el diccionario de datos como un DataFrame y guarda el dataset limpio
# Ejemplo de estructura:
# diccionario = pd.DataFrame([
#     {"columna": "salario_mensual", "tipo": "float", "descripcion": "Salario mensual en pesos",
#      "transformacion": "ninguna", "rango": "10000-90000"},
#     {"columna": "salario_z", "tipo": "float", "descripcion": "Salario estandarizado",
#      "transformacion": "Z-score", "rango": "media 0, desv 1"},
#     # ... completa TODAS las columnas
# ])
# print(diccionario.to_string(index=False))
# df_transformado.to_csv("dataset_limpio.csv", index=False)
# diccionario.to_csv("diccionario_datos.csv", index=False)


---
## Cierre y entrega

Dominaste las cinco técnicas de transformación de datos: **normalización, estandarización, codificación categórica (ordinal, one-hot, label), feature engineering y manejo de fechas**. Además construiste un **pipeline reproducible** y documentaste un dataset con su **diccionario de datos**, tal como se hace en un proyecto profesional.

### Entrega en Google Classroom

1. Verifica que **todas las celdas corran sin error** (Entorno de ejecución → Ejecutar todo).
2. Confirma que respondiste las **50 preguntas ✍️** y los **9 retos de código**.
3. Adjunta el **dataset limpio** (`dataset_limpio.csv`) y el **diccionario de datos**.
4. **Archivo → Guardar una copia en Drive** y comparte el enlace en Classroom.

### Rúbrica

| Criterio | Puntos |
|---|---|
| Todas las celdas se ejecutan correctamente | 10 |
| Retos de código resueltos (9) | 25 |
| Respuestas a las 50 preguntas ✍️ | 35 |
| Pipeline reproducible funcional | 10 |
| Dataset limpio + diccionario de datos | 15 |
| Orden y documentación | 5 |
| **Total** | **100** |

---
*Universidad de Especialidades UNE · Plantel Centro · Optativa III: Ciencia de Datos*
*Datasets: empleados.csv, amazon.csv*
